# Expert policies

> Scripted MultiGrid policies with access to the true env state, e.g. to collect expert data.

In [ ]:
#| default_exp envs.multigrid.expert

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from collections import deque

import numpy as np

from stable_marl.policy.base import ExpertPolicy
from stable_marl.envs.multigrid.base import MultiGridEnv
from stable_marl.envs.multigrid.core.actions import Action
from stable_marl.envs.multigrid.core.constants import Direction
from stable_marl.envs.multigrid.core.world_object import Goal

In [ ]:
#| export
class GoToGoalPolicy(ExpertPolicy):
    """
    Navigation expert (e.g. for the Empty and FindGoal envs): every agent follows a shortest
    path to the nearest goal, over the cells it can walk on, turning towards the next cell
    and then moving forward.

    Agents without a path to a goal (or already terminated) turn left. Other agents are not
    treated as obstacles.
    """
    def act(self, env: MultiGridEnv) -> np.ndarray:
        distance = goal_distances(env)
        return np.array([self._agent_action(env, agent, distance) for agent in env.agents])

    @staticmethod
    def _agent_action(env: MultiGridEnv, agent, distance: np.ndarray) -> int:
        x, y = (int(v) for v in agent.state.pos)
        if agent.state.terminated or not np.isfinite(distance[x, y]) or distance[x, y] == 0:
            return int(Action.left)
        current = int(agent.state.dir)
        # directions towards a neighbour one step closer to a goal, preferring the current one
        closer = [d for d in Direction if _distance_at(distance, (x, y) + d.to_vec()) == distance[x, y] - 1]
        if current in closer:
            return int(Action.forward)
        if (current + 1) % 4 in closer:
            return int(Action.right)
        return int(Action.left)


def _distance_at(distance: np.ndarray, pos) -> float:
    x, y = pos
    w, h = distance.shape
    return distance[x, y] if 0 <= x < w and 0 <= y < h else np.inf


def goal_distances(env: MultiGridEnv) -> np.ndarray:
    "Number of steps from every cell to the nearest goal, walking on overlappable cells (inf: unreachable)."
    distance = np.full((env.width, env.height), np.inf)
    queue = deque()
    for x in range(env.width):
        for y in range(env.height):
            if isinstance(env.grid.get(x, y), Goal):
                distance[x, y] = 0
                queue.append((x, y))
    while queue:
        x, y = queue.popleft()
        for d in Direction:
            nx, ny = (int(v) for v in (x, y) + d.to_vec())
            if not (0 <= nx < env.width and 0 <= ny < env.height) or np.isfinite(distance[nx, ny]):
                continue
            cell = env.grid.get(nx, ny)
            if cell is None or cell.can_overlap():
                distance[nx, ny] = distance[x, y] + 1
                queue.append((nx, ny))
    return distance

### Tests

In [ ]:
from stable_marl.envs import make

env = make('MultiGrid-Empty-8x8-v0', agents=1)
env.reset(seed=0)
policy = GoToGoalPolicy()
for t in range(100):
    obs, rew, term, trunc, _ = env.step(dict(enumerate(policy.act(env))))
    if term[0]: break
assert term[0] and rew[0] > 0 and t < 2 * (env.width + env.height)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()